#Silver Layer - This layer is for Data Munging, Enrichment, customization and curation.  We can store the data as per the Business needs

In [1]:
#Run the below notebook for invoking the function to this notebook
%run "/notebook/PysparkCode/1_Generic_function.ipynb"

In [2]:
#Applying Strict Schema to the DataFrame, so creating schema
from pyspark.sql.types import *

schema_under = StructType([
        StructField("Job_ID", IntegerType(), True),
        StructField("Agency", StringType(), True),
        StructField("Posting_Type", StringType(), True),
        StructField("No_Of_Positions", IntegerType(), True),
        StructField("Business_Title", StringType(), True),
        StructField("Civil_Service_Title", StringType(), True),
        StructField("Title_Code_No", StringType(), True),
        StructField("Level", StringType(), True),
        StructField("Job_Category", StringType(), True),
        StructField("Full-Time/Part-Time_indicator", StringType(), True),
        StructField("Salary_Range_From", DoubleType(), True),
        StructField("Salary_Range_To", DoubleType(), True),
        StructField("Salary_Frequency", StringType(), True),
        StructField("Work_Location", StringType(), True),
        StructField("Division/Work_Unit", StringType(), True),
        StructField("Job_Description", StringType(), True),
        StructField("Minimum_Qual_Requirements", StringType(), True),
        StructField("Preferred_Skills", StringType(), True),
        StructField("Additional_Information", StringType(), True),
        StructField("To_Apply", StringType(), True),
        StructField("Hours/Shift", StringType(), True),
        StructField("Work_Location_1", StringType(), True),
        StructField("Recruitment_Contact", StringType(), True),
        StructField("Residency_Requirement", StringType(), True),
        StructField("Posting_Date", StringType(), True),
        StructField("Post_Until", StringType(), True),
        StructField("Posting_Updated", StringType(), True),
        StructField("Process_Date", StringType(), True),
        StructField("Corrupt_rows", StringType(), True)
    ])

#Applying Strict Schema to the DataFrame, so creating schema

from pyspark.sql.types import *

Struct_schema = StructType([
        StructField("Job ID", IntegerType(), True),
        StructField("Agency", StringType(), True),
        StructField("Posting Type", StringType(), True),
        StructField("# Of Positions", IntegerType(), True),
        StructField("Business Title", StringType(), True),
        StructField("Civil Service Title", StringType(), True),
        StructField("Title Code No", StringType(), True),
        StructField("Level", StringType(), True),
        StructField("Job Category", StringType(), True),
        StructField("Full-Time/Part-Time indicator", StringType(), True),
        StructField("Salary Range From", DoubleType(), True),
        StructField("Salary Range To", DoubleType(), True),
        StructField("Salary Frequency", StringType(), True),
        StructField("Work Location", StringType(), True),
        StructField("Division/Work Unit", StringType(), True),
        StructField("Job Description", StringType(), True),
        StructField("Minimum Qual Requirements", StringType(), True),
        StructField("Preferred Skills", StringType(), True),
        StructField("Additional Information", StringType(), True),
        StructField("To Apply", StringType(), True),
        StructField("Hours/Shift", StringType(), True),
        StructField("Work Location 1", StringType(), True),
        StructField("Recruitment Contact", StringType(), True),
        StructField("Residency Requirement", StringType(), True),
        StructField("Posting Date", StringType(), True),
        StructField("Post Until", StringType(), True),
        StructField("Posting Updated", StringType(), True),
        StructField("Process Date", StringType(), True),
        StructField("Corrupt_rows", StringType(), True)
    ])

##Data Munging

In [ ]:
#Reads the file in orc format and do count level validation
df=Active_Munging("/dataset/Bronze",Struct_schema,"Corrupt_rows","Job ID")
print("Total Number of records",df.count())
print("Total Number of De-Duplicated records",df.distinct().count())
#df.show()

##Data Enrichment - Pre processing the data

In [ ]:
#df.show()
silver_df=data_enrichment(df,["Posting Date","Post Until","Posting Updated","Process Date"],"Job ID")
#silver_df.show(12)

##Data Curation - transformation, filtering<br>
List of KPIs to be resolved:<br>
Whats the number of jobs posting per category (Top 10)?<br>
Whats the salary distribution per job category?<br>
Is there any correlation between the higher degree and the salary?<br>
Whats the job posting having the highest salary per agency?<br>
Whats the job positings average salary per agency for the last 2 years?<br>
What are the highest paid skills in the US market?<br>

In [ ]:
#Whats the number of jobs posting per category (Top 10)?
def No_of_jobs(df):
    df1=df.groupBy(upper(col("Job Category")).alias("Job Category")).count().orderBy("Count",ascending=False).limit(10)
    return df1

No_of_jobs(silver_df).show()

In [ ]:
#Whats the salary distribution per job category?
def Salary_distribution(df):
    SF_LIST=df.select(upper(col("Job Category")).alias("Job Category"),upper(col("Salary Frequency")).alias("Salary Frequency")).distinct()
    return SF_LIST

Salary_distribution(silver_df).show(truncate=False)

##Data Customization

In [ ]:
#Is there any correlation between the higher degree and the salary?
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re

# Define UDF to extract education level from text
def extract_education_level(qual):
    if qual is None:
        return 0
    
    text = qual.lower()
    
# Define education level patterns
    if any(term in text for term in ['phd', 'doctorate', 'doctoral']):
        return 4
    elif any(term in text for term in ['master', 'mba', 'ma degree', 'ms degree', 'graduate degree']):
        return 3
    elif any(term in text for term in ['bachelor', 'baccalaureate', 'ba degree', 'bs degree', 'undergraduate']):
        return 2
    elif any(term in text for term in ['high school', 'diploma', 'ged', 'h.s.']):
        return 1
    else:
        return 0
    
# Calculate Average Salary and annual_salary

df_with_AvgSal = silver_df.withColumn("Average_Salary",((col("Salary Range From") + col("Salary Range To")) / 2)) \
                   .withColumn("Annual_salary",when(col("Salary Frequency") == "Annual", col("Average_Salary"))\
                                 .when(col("Salary Frequency") == "Hourly", col("Average_Salary") * 2080)\
                                 .when(col("Salary Frequency") == "Daily", col("Average_Salary") * 260)\
                                 .otherwise(None))


# Register UDF
extract_education_udf = udf(extract_education_level)



#write to table

df_with_AvgSal.write.orc("/dataset/Silver",mode="overwrite")

df_Enhanced=df_with_AvgSal.withColumn("Education_Level",extract_education_udf(col("Minimum Qual Requirements")))

#df_Enhanced.select("Education_Level","Annual_salary").show()
correlation = df_Enhanced.select(corr("Education_Level","Annual_salary"))
correlation.show()
#df_Enhanced.summary().show()